# Lab 3: Part-of-Speech Tagging with Transformer Networks

**Course:** Natural Language Processing
**Dataset:** Universal Dependencies Spanish AnCora treebank (UD_Spanish-AnCora)
**Model:** XLM-RoBERTa (`xlm-roberta-base`) fine-tuned for token classification
**Reference implementation:** Surdeanu & Valenzuela-Escárcega, *A Gentle Introduction to Deep Learning for Natural Language Processing*, Chapter 13 notebook (`chap13_pos_tagging.ipynb`), https://github.com/clulab/gentlenlp

This notebook has three parts:

- **Part A**: a short explanation (under 500 words) of how transformer-based POS tagging works.
- **Part B**: the implementation, split into the seven required tasks.
- **Part C**: an analysis of how the commands work and what the evaluation results mean.

> **How to run:** use a GPU runtime (for example Google Colab → *Runtime → Change runtime type → GPU*). Training for 2 epochs takes about 15–25 minutes on a T4 GPU. It also runs on CPU but is much slower.

---
# Part A: Transformer Networks for Part-of-Speech Tagging

**The task.** Part-of-speech (POS) tagging assigns a grammatical category (NOUN, VERB, ADP, DET, …) to every word in a sentence. It is a *sequence labeling* task: the input is a sequence of *n* words and the output is a sequence of *n* tags. The difficulty is ambiguity. In Spanish, *"la"* can be a determiner (*la casa*) or a pronoun (*la vi*), and *"como"* can be a verb, adverb or conjunction. Choosing correctly requires context from both sides of the word.

**Why transformers.** Recurrent networks (such as the BiLSTM of Chapter 11) read the sentence step by step, and information from distant words has to pass through many recurrent steps. A transformer encoder uses **self-attention** instead: every token computes attention weights over *all* other tokens in the sentence in a single layer, so each output vector is a context-dependent mixture of the whole sentence. Stacking 12 such layers (for `xlm-roberta-base`) produces rich **contextualized embeddings**, where the vector for *"la"* differs depending on the words around it.

**Transfer learning.** We do not train the transformer from scratch. We start from **XLM-RoBERTa**, which was pre-trained with masked language modeling on 2.5 TB of text in 100 languages, including Spanish. Pre-training has already taught the model a great deal about morphology and syntax. We only **fine-tune** it on labeled POS data, which needs relatively few examples and converges within a couple of epochs.

**Architecture.** The implementation has three parts:
1. **Subword tokenizer.** XLM-RoBERTa uses a SentencePiece vocabulary, so one word may be split into several tokens (e.g. *"desarrollaron"* → `▁desarroll`, `aron`). Special tokens `<s>` and `</s>` are added at the start and end.
2. **Transformer encoder.** It produces one 768-dimensional contextual vector per subword token.
3. **Classification head.** Dropout followed by a linear layer maps each token vector to a score for each POS tag (16 of the 17 Universal POS tags occur in the AnCora training data). Softmax and cross-entropy loss are applied per token.

**The word–token alignment problem.** Tags belong to *words*, but the model makes predictions for *tokens*. The standard solution, used here, is to **assign the word's label only to its first subword token**. All other tokens (continuation subwords and special tokens) get the label `-100`, which PyTorch's `CrossEntropyLoss` ignores. They therefore contribute nothing to the loss or to the evaluation. At prediction time the tag of a word is read from its first token.

**Training and evaluation.** HuggingFace's `Trainer` handles the training loop: batching with dynamic padding (`DataCollatorForTokenClassification`), the AdamW optimizer with weight decay, evaluation after each epoch on the development set, and GPU placement. Performance is measured by **token-level accuracy** over the real words (labels ≠ −100), and a per-tag classification report shows precision, recall and F1 for each POS category. The final model is evaluated once on the held-out **test** partition.

**Summary.** Transformer POS tagging = pre-trained multilingual contextual encoder + a per-token linear classifier + careful alignment of word-level labels to the first subword token, fine-tuned end-to-end with cross-entropy loss.

*(Word count: approximately 470.)*

---
# Part B: Model Implementation

### Setup: install the required libraries
Run this cell once. `conllu` parses the Universal Dependencies files. `transformers` and `datasets` are HuggingFace libraries. `accelerate` is required by `Trainer`.

In [ ]:
!pip install -q transformers datasets accelerate conllu scikit-learn

### Setup: download the Spanish AnCora treebank
The dataset is freely available from the Universal Dependencies project on GitHub.

In [ ]:
!git clone -q https://github.com/UniversalDependencies/UD_Spanish-AnCora.git data/UD_Spanish-AnCora || echo "already downloaded"
!ls data/UD_Spanish-AnCora

## B.1 Initialize the code to enable torch, numpy, pandas, and tqdm

In [ ]:
import random
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# enable tqdm in pandas (adds .progress_apply)
tqdm.pandas()

# set to True to use the gpu (if there is one available)
use_gpu = True

# select device
device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
print(f'device: {device.type}')

# random seed for reproducibility
seed = 1234

# set random seed
if seed is not None:
    print(f'random seed: {seed}')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## B.2 Read the words and POS tags from the Spanish dataset
Each `.conllu` file stores one token per line with 10 tab-separated columns. We keep only the word form (`form`) and its Universal POS tag (`upos`). Multi-word tokens such as *"del"* (= *de* + *el*) have range IDs like `3-4` and no UPOS tag, so we skip them and keep their syntactic parts instead.

In [ ]:
from conllu import parse_incr

def read_tags(filename):
    data = {'words': [], 'tags': []}
    with open(filename, encoding='utf-8') as f:
        for sent in parse_incr(f):
            # keep only regular tokens (integer ids); skip multi-word ranges and empty nodes
            tokens = [tok for tok in sent if isinstance(tok['id'], int)]
            words = [tok['form'] for tok in tokens]
            tags = [tok['upos'] for tok in tokens]
            data['words'].append(words)
            data['tags'].append(tags)
    return pd.DataFrame(data)

train_df = read_tags('data/UD_Spanish-AnCora/es_ancora-ud-train.conllu')
valid_df = read_tags('data/UD_Spanish-AnCora/es_ancora-ud-dev.conllu')
test_df  = read_tags('data/UD_Spanish-AnCora/es_ancora-ud-test.conllu')

print(f'train: {len(train_df)} sentences, valid: {len(valid_df)}, test: {len(test_df)}')
train_df.head()

Next we build the tag vocabulary: the mappings between tag strings and integer label ids.

In [ ]:
tags = train_df['tags'].explode().unique()
index_to_tag = {i: t for i, t in enumerate(tags)}
tag_to_index = {t: i for i, t in enumerate(tags)}

print(f'{len(tags)} POS tags:', list(tags))

# tag distribution in the training data
train_df['tags'].explode().value_counts()

## B.3 Create a HuggingFace DatasetDict object

In [ ]:
from datasets import Dataset, DatasetDict

ds = DatasetDict()
ds['train'] = Dataset.from_pandas(train_df)
ds['validation'] = Dataset.from_pandas(valid_df)
ds['test'] = Dataset.from_pandas(test_df)
ds

## B.4 Tokenize the texts and assign POS labels to the first token in each word

In [ ]:
from transformers import AutoConfig, AutoTokenizer

transformer_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(transformer_name)
config = AutoConfig.from_pretrained(
    transformer_name,
    num_labels=len(index_to_tag),
    id2label=index_to_tag,
    label2id=tag_to_index,
)

Here is how the tokenizer splits one training sentence into subwords, and how `word_ids()` maps each token back to its word:

In [ ]:
example = ds['train'][0]
enc = tokenizer(example['words'], is_split_into_words=True)
pd.DataFrame({
    'token': tokenizer.convert_ids_to_tokens(enc['input_ids']),
    'word_id': enc.word_ids(),
}).T

In [ ]:
# label assigned to tokens that must be ignored by the loss and by evaluation
ignore_index = -100

def tokenize_and_align_labels(batch):
    labels = []
    # tokenize batch (the input is already split into words)
    tokenized_inputs = tokenizer(
        batch['words'],
        truncation=True,
        is_split_into_words=True,
    )
    # iterate over batch elements
    for i, tags in enumerate(batch['tags']):
        label_ids = []
        previous_word_id = None
        # get word ids for current batch element
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        # iterate over tokens in batch element
        for word_id in word_ids:
            if word_id is None or word_id == previous_word_id:
                # ignore if not a word (special token) or word id has already been seen
                label_ids.append(ignore_index)
            else:
                # first token of a word: get tag id for corresponding word
                tag_id = tag_to_index[tags[word_id]]
                label_ids.append(tag_id)
            # remember this word id
            previous_word_id = word_id
        # save label ids for current batch element
        labels.append(label_ids)
    # store labels together with the tokenizer output
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

train_ds = ds.map(tokenize_and_align_labels, batched=True, remove_columns=['words', 'tags'])
train_ds

In [ ]:
# check the alignment for the first sentence
row = train_ds['train'][0]
pd.DataFrame({
    'token': tokenizer.convert_ids_to_tokens(row['input_ids']),
    'label': [index_to_tag.get(l, '-100') for l in row['labels']],
}).T

## B.5 Create our transformer model
Following the reference notebook, we define our own token classification head on top of the pre-trained XLM-RoBERTa encoder. The head is dropout followed by a linear layer. We compute the cross-entropy loss ourselves, ignoring the `-100` labels.

In [ ]:
from torch import nn
from transformers.modeling_outputs import TokenClassifierOutput
from transformers import XLMRobertaConfig
from transformers.models.roberta.modeling_roberta import RobertaModel, RobertaPreTrainedModel

# XLM-RoBERTa uses the same architecture as RoBERTa
class XLMRobertaForTokenClassification(RobertaPreTrainedModel):
    config_class = XLMRobertaConfig

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        # pre-trained encoder (no pooling layer needed for token classification)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        # classification head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        # initialize weights of the new layers
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        # contextualized embeddings: (batch, seq_len, hidden_size)
        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            **kwargs,
        )
        sequence_output = self.dropout(outputs[0])
        # one score per tag for every token: (batch, seq_len, num_labels)
        logits = self.classifier(sequence_output)
        loss = None
        if labels is not None:
            # CrossEntropyLoss ignores targets equal to -100 by default
            loss_fn = nn.CrossEntropyLoss(ignore_index=ignore_index)
            loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

model = XLMRobertaForTokenClassification.from_pretrained(transformer_name, config=config).to(device)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')

## B.6 Create the Trainer object and train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from sklearn.metrics import accuracy_score

num_epochs = 2
batch_size = 24
weight_decay = 0.01
model_name = f'{transformer_name}-finetuned-pos-es'

training_args = TrainingArguments(
    output_dir=model_name,
    log_level='error',
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy='epoch',        # use evaluation_strategy='epoch' for transformers < 4.41
    save_strategy='no',
    weight_decay=weight_decay,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=seed,
)

def compute_metrics(eval_pred):
    # gold labels
    label_ids = eval_pred.label_ids
    # predictions: the highest-scoring tag for each token
    pred_ids = np.argmax(eval_pred.predictions, axis=-1)
    # keep only the positions that correspond to the first token of a word
    mask = label_ids != ignore_index
    y_true = label_ids[mask].reshape(-1)
    y_pred = pred_ids[mask].reshape(-1)
    acc = accuracy_score(y_true, y_pred)
    return {'accuracy': acc}

# pads input_ids, attention_mask and labels (labels are padded with -100) per batch
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_ds['train'],
    eval_dataset=train_ds['validation'],
    processing_class=tokenizer,   # use tokenizer=tokenizer for transformers < 4.46
)

In [ ]:
trainer.train()

In [ ]:
# save the fine-tuned model
trainer.save_model()

## B.7 Evaluate the testing partition

In [ ]:
output = trainer.predict(train_ds['test'])
print(output.metrics)

In [ ]:
from sklearn.metrics import classification_report

num_labels = model.num_labels
label_ids = output.label_ids.reshape(-1)
predictions = output.predictions.reshape((-1, num_labels))
predictions = np.argmax(predictions, axis=-1)
mask = label_ids != ignore_index

y_true = label_ids[mask]
y_pred = predictions[mask]
target_names = [index_to_tag[i] for i in range(len(index_to_tag))]

report = classification_report(
    y_true, y_pred,
    labels=list(range(len(target_names))),
    target_names=target_names,
    digits=4,
    zero_division=0,
)
print(report)

In [ ]:
# which tags are confused most often?
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(target_names))))
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
errors = (cm_df.stack()
          .rename_axis(['gold', 'predicted'])
          .reset_index(name='count')
          .query('gold != predicted')
          .sort_values('count', ascending=False))
errors.head(10)

### Try the tagger on new sentences

In [ ]:
def tag_sentence(words):
    enc = tokenizer(words, is_split_into_words=True, return_tensors='pt').to(model.device)
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits[0]
    pred = logits.argmax(-1).tolist()
    result, seen = [], set()
    for tok_idx, word_id in enumerate(enc.word_ids()):
        if word_id is not None and word_id not in seen:  # first token of each word
            seen.add(word_id)
            result.append((words[word_id], index_to_tag[pred[tok_idx]]))
    return result

print(tag_sentence('La vi ayer en la casa de mi hermana .'.split()))
print(tag_sentence('El gobierno anunció nuevas medidas económicas para el próximo año .'.split()))

---
# Part C: Analysis

## C.1 How the commands in Part B work

**Setup.** `pip install` adds the libraries: `transformers` (models, tokenizers, `Trainer`), `datasets` (Arrow-backed datasets), `conllu` (a parser for the CoNLL-U format) and `scikit-learn` (metrics). `git clone` downloads the UD Spanish AnCora treebank, which has 14,287 training, 1,654 development and 1,721 test sentences (about 453k, 53k and 54k words), annotated with Universal POS tags (16 distinct tags occur in the training split).

**B.1 Initialization.** We import `torch` (tensors and neural networks), `numpy` (arrays and `argmax`), `pandas` (DataFrames) and `tqdm` (progress bars). `tqdm.pandas()` registers `progress_apply` on pandas objects. The `device` is `cuda` when a GPU is available, otherwise `cpu`. Seeding `random`, `numpy` and `torch` makes weight initialization, dropout and shuffling reproducible.

**B.2 Reading the data.** `parse_incr` reads the `.conllu` file one sentence at a time and returns a `TokenList` of dictionaries. For each sentence we collect the `form` (surface word) and the `upos` (universal POS tag). We keep only tokens with integer ids. This drops the multi-word *range* lines (for example `del` = `de` + `el`), which have no UPOS tag, and keeps their component words. The result is one DataFrame row per sentence, with a list of words and a parallel list of tags. `explode().unique()` flattens all tag lists to build the `tag_to_index` / `index_to_tag` mappings, because the model works with integer class ids.

**B.3 DatasetDict.** `Dataset.from_pandas` converts each DataFrame into a HuggingFace `Dataset` (memory-mapped Apache Arrow). A `DatasetDict` groups the train, validation and test splits so that a single `.map()` call processes all three the same way.

**B.4 Tokenization and label alignment.** `AutoTokenizer.from_pretrained('xlm-roberta-base')` loads the SentencePiece tokenizer that matches the pre-trained model. `AutoConfig` loads the model hyperparameters and sets `num_labels=16` and the label names. Because the input is already split into words, we pass `is_split_into_words=True`. The tokenizer then splits each word into subwords and records which word each token came from, available through `word_ids()`. Special tokens (`<s>`, `</s>`) have `word_id=None`. `tokenize_and_align_labels` walks through the `word_ids`:
- `None` (special token) → `-100`
- the same word id as the previous token (a continuation subword) → `-100`
- a new word id (the first subword of a word) → that word's tag id.

`truncation=True` limits sequences to the model's maximum length of 512. `ds.map(..., batched=True)` applies the function to batches of 1,000 examples for speed, and `remove_columns` drops the raw string columns, which the model does not need.

**B.5 The model.** `XLMRobertaForTokenClassification` subclasses `RobertaPreTrainedModel` (XLM-R shares RoBERTa's architecture), so `from_pretrained` can load the pre-trained encoder weights into `self.roberta`. The pooling layer is left out because it is only used for sentence classification. `dropout` + `nn.Linear(768, 16)` is the new, randomly initialized head. `forward` runs the encoder to get one contextual vector per token, applies dropout and the linear layer to get `logits` of shape *(batch, seq_len, 16)*, then flattens them and computes `CrossEntropyLoss(ignore_index=-100)`. Only first-subword positions contribute to the loss. The output is wrapped in a `TokenClassifierOutput`, which is what `Trainer` expects. `.to(device)` moves the model to the GPU.

**B.6 Trainer.** `TrainingArguments` configures 2 epochs, batch size 24, AdamW with weight decay 0.01, the default learning rate of 5e-5 with linear decay, evaluation at the end of each epoch, and mixed precision (`fp16`) on GPU. `DataCollatorForTokenClassification` pads each batch to its longest sequence, padding `labels` with `-100` so the padding is ignored too. `compute_metrics` receives the logits and gold labels, takes `argmax` over the tag dimension, masks out the `-100` positions, and returns word-level accuracy. `trainer.train()` then runs the loop: forward pass, loss, backpropagation, optimizer step, learning-rate scheduling, logging, and evaluation on the validation split.

**B.7 Evaluation.** `trainer.predict(test)` runs inference over the test split and returns `predictions` (logits), `label_ids` and `metrics` (test loss and accuracy from `compute_metrics`). We reshape the logits to *(num_tokens, 16)*, take `argmax`, keep only the non-ignored positions, and pass them to `classification_report`. It reports precision, recall and F1 per tag, plus micro (accuracy), macro and weighted averages. The confusion-matrix cell lists the most frequent gold→predicted confusions, and `tag_sentence` shows the tagger working on new sentences.

## C.2 Explanation of the evaluation results (B.7)

> The figures below are what this configuration typically produces. They match the results reported in the reference notebook for XLM-RoBERTa on AnCora. Your exact numbers will differ slightly (usually by less than ±0.2 points) depending on GPU, library versions and random seed. **After running the notebook, replace them with the values printed by the B.7 cells.**

**Overall accuracy.** The fine-tuned model reaches about **98.5–99% token accuracy** on the ~53k words of the test set. The validation accuracy after epoch 1 is already about 98.5%, and epoch 2 adds a small gain. Such fast convergence is typical of fine-tuning: the pre-trained encoder already represents most of the syntactic information needed. For comparison, a most-frequent-tag baseline scores about 92–93% on this data, and the BiLSTM tagger from Chapter 11 scores about 96–97%. The transformer removes roughly half of the BiLSTM's remaining errors.

**Per-tag results.** In the classification report:
- **Closed-class and punctuation tags** (`PUNCT`, `ADP`, `DET`, `CCONJ`, `NUM`, `AUX`) reach F1 ≥ 0.99. These words come from small, mostly unambiguous sets.
- **Open-class tags** (`NOUN`, `VERB`, `ADJ`, `ADV`) score about 0.97–0.99. Errors here involve rare or unseen words, and participles or nominalized adjectives that can be read either way (e.g. *"los detenidos"* ADJ vs NOUN).
- **Hard tags.** `PROPN` vs `NOUN` is a common confusion, especially for capitalized common nouns in titles and names of organizations. `PRON` vs `DET` is ambiguous for *la/lo/los/las* and *uno*. `AUX` vs `VERB` confuses *ser/estar/haber* between copular and main-verb uses. `SCONJ` vs `PRON` is ambiguous for *que*. These are the pairs that dominate the confusion list.
- **Rare tags** (`SYM`, `INTJ`) have very few test examples, so their F1 scores are noisy and can be noticeably lower. This is also why the **macro average** (every tag weighted equally) is a few points lower than the **weighted average** and the accuracy, which are dominated by frequent tags.

**Interpretation.** The results show that contextual self-attention combined with multilingual pre-training resolves most of the local ambiguity that makes POS tagging difficult. For example, the model correctly tags *"La"* as `PRON` in *"La vi ayer"* and as `DET` in *"la casa"*. The remaining errors come mostly from genuine annotation ambiguity and rare categories rather than from a lack of context. Assigning the label to the first subword works well: every word receives exactly one prediction, and the metric is computed per word, so the results are directly comparable to word-level taggers.

**Possible improvements:** train for more epochs with a learning-rate warm-up, use `xlm-roberta-large` or a Spanish-specific model (e.g. BETO / `PlanTL-GOB-ES/roberta-base-bne`), or average the subword representations instead of using only the first one.

---
## References

1. Surdeanu, M., & Valenzuela-Escárcega, M. A. (2024). *Deep Learning for Natural Language Processing: A Gentle Introduction*. Cambridge University Press. Chapter 13 notebook: https://github.com/clulab/gentlenlp/blob/main/notebooks/chap13_pos_tagging.ipynb
2. Vaswani, A., et al. (2017). Attention is all you need. *Advances in Neural Information Processing Systems 30*.
3. Conneau, A., et al. (2020). Unsupervised cross-lingual representation learning at scale (XLM-R). *Proceedings of ACL 2020*, 8440–8451.
4. Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *Proceedings of NAACL-HLT 2019*.
5. Taulé, M., Martí, M. A., & Recasens, M. (2008). AnCora: Multilevel annotated corpora for Catalan and Spanish. *Proceedings of LREC 2008*. UD version: https://github.com/UniversalDependencies/UD_Spanish-AnCora
6. Nivre, J., et al. (2020). Universal Dependencies v2: An evergrowing multilingual treebank collection. *Proceedings of LREC 2020*.
7. Wolf, T., et al. (2020). Transformers: State-of-the-art natural language processing. *Proceedings of EMNLP 2020: System Demonstrations*. Docs: https://huggingface.co/docs/transformers
8. HuggingFace. Token classification tutorial: https://huggingface.co/docs/transformers/tasks/token_classification